In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gc
import os
import sys

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

In [ ]:
## load imp genes and their Entrez Gene ID

imp_gene_entrez_ID = pd.read_csv("Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/imp_genes_entrez_ID.csv")

In [ ]:
imp_entrez_ID = list(imp_gene_entrez_ID['EntrezID'])

In [ ]:
rna_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_luad.csv'))
rna_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\gene_exp\with_common_patients_processed', 'csv_rna_common_lusc.csv'))

rna_luad_xena['label'] = 1
rna_lusc_xena['label'] = 0
df_rna_xena = pd.concat([rna_luad_xena, rna_lusc_xena], axis=0)

cnv_luad_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_luad.csv'))
cnv_lusc_xena = pd.read_csv(os.path.join('Z:\multiomics based manuscript\datasets\cnv\with_common_patients_processed', 'csv_cnv_common_lusc.csv'))

cnv_luad_xena['label'] = 1
cnv_lusc_xena['label'] = 0
df_cnv_xena = pd.concat([cnv_luad_xena, cnv_lusc_xena], axis=0)

In [ ]:
# ## df.columns.intersection keeps only imp_gene list values as columns

# df_lusc_rna = df_lusc_rna[df_lusc_rna.columns.intersection(imp_entrez_ID)]
# df_lusc_cnv = df_lusc_cnv[df_lusc_cnv.columns.intersection(imp_entrez_ID)]

# df_luad_rna = df_luad_rna[df_luad_rna.columns.intersection(imp_entrez_ID)]
# df_luad_cnv = df_luad_cnv[df_luad_cnv.columns.intersection(imp_entrez_ID)]


In [ ]:
df_rna = df_rna_xena
df_cnv = df_cnv_xena

In [ ]:
df_rna.reset_index(drop=True, inplace=True)
df_cnv.reset_index(drop=True, inplace=True)

In [ ]:
## perform classification using XGBoost
def perform_classification(df, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    model = XGBClassifier(device='cuda')

    y_pred_val = np.zeros_like(y)
    y_proba_val = np.zeros_like(y, dtype=float)

    y_pred_train = np.zeros_like(y)
    y_proba_train = np.zeros_like(y, dtype=float)

    for train_idx, val_idx in skfold.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh model for each fold
        model = XGBClassifier(device='cuda')
        model.fit(X_train, y_train)

        # --- Validation-fold predictions ---
        y_pred_val[val_idx] = model.predict(X_val)
        y_proba_val[val_idx] = model.predict_proba(X_val)[:, 1]

        # --- Training-fold predictions ---
        y_pred_train[train_idx] = model.predict(X_train)
        y_proba_train[train_idx] = model.predict_proba(X_train)[:, 1]

    # Metrics (against validation predictions)
    accuracy_train = accuracy_score(y, y_pred_train)
    accuracy_val = accuracy_score(y, y_pred_val)
    auroc = roc_auc_score(y, y_proba_val)
    cm = confusion_matrix(y, y_pred_val)

    results = {
        "accuracy_train": accuracy_train,
        "accuracy_val": accuracy_val,
        "auroc": auroc,
        "cm": cm,
        "y_pred": y_pred_val,
        "y_proba": y_proba_val,
        "y_true": y
    }

    return results

In [ ]:
## perform classification using MLP
def perform_classification_mlp(df, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    model = MLPClassifier(random_state=random_state)

    y_pred_val = np.zeros_like(y)
    y_proba_val = np.zeros_like(y, dtype=float)

    y_pred_train = np.zeros_like(y)
    y_proba_train = np.zeros_like(y, dtype=float)

    for train_idx, val_idx in skfold.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh model for each fold
        model = MLPClassifier(random_state=random_state)
        model.fit(X_train, y_train)

        # --- Validation-fold predictions ---
        y_pred_val[val_idx] = model.predict(X_val)
        y_proba_val[val_idx] = model.predict_proba(X_val)[:, 1]

        # --- Training-fold predictions ---
        y_pred_train[train_idx] = model.predict(X_train)
        y_proba_train[train_idx] = model.predict_proba(X_train)[:, 1]

    # Metrics (against validation predictions)
    accuracy_train = accuracy_score(y, y_pred_train)
    accuracy_val = accuracy_score(y, y_pred_val)
    auroc = roc_auc_score(y, y_proba_val)
    cm = confusion_matrix(y, y_pred_val)

    results = {
        "accuracy_train": accuracy_train,
        "accuracy_val": accuracy_val,
        "auroc": auroc,
        "cm": cm,
        "y_pred": y_pred_val,
        "y_proba": y_proba_val,
        "y_true": y
    }

    return results

In [ ]:
## perform classification using SVC
def perform_classification_svc(df, random_state):
    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    model = SVC(probability=True)

    y_pred_val = np.zeros_like(y)
    y_proba_val = np.zeros_like(y, dtype=float)

    y_pred_train = np.zeros_like(y)
    y_proba_train = np.zeros_like(y, dtype=float)

    for train_idx, val_idx in skfold.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh model for each fold
        model = SVC(probability=True)
        model.fit(X_train, y_train)

        # --- Validation-fold predictions ---
        y_pred_val[val_idx] = model.predict(X_val)
        y_proba_val[val_idx] = model.predict_proba(X_val)[:, 1]

        # --- Training-fold predictions ---
        y_pred_train[train_idx] = model.predict(X_train)
        y_proba_train[train_idx] = model.predict_proba(X_train)[:, 1]

    # Metrics (against validation predictions)
    accuracy_train = accuracy_score(y, y_pred_train)
    accuracy_val = accuracy_score(y, y_pred_val)
    auroc = roc_auc_score(y, y_proba_val)
    cm = confusion_matrix(y, y_pred_val)

    results = {
        "accuracy_train": accuracy_train,
        "accuracy_val": accuracy_val,
        "auroc": auroc,
        "cm": cm,
        "y_pred": y_pred_val,
        "y_proba": y_proba_val,
        "y_true": y
    }

    return results

In [ ]:
def perform_tabnet(df, random_state):
    """
    Check the performance of TabNet
    """
    
    accuracies = []
    accuracies_train = []
    aurocs = []
    conf_mats = []
    y_trues = []
    y_preds_proba = []

    X = df.drop(columns=['label'])
    y = df['label']
    X = X.to_numpy()
    y = y.to_numpy()
    
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    
    for train_idx, test_idx in skfold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
    
        tabnet_model = TabNetClassifier(
            device_name = 'cuda',
            seed = random_state,
            verbose = 0
        )
        tabnet_model.fit(
            X_train, y_train,
            eval_set = [(X_test, y_test)],
            # max_epochs = 3,
            eval_metric = ['auc', 'accuracy'],
            batch_size = 512,
            patience = 0
        )
    
        y_pred_val = tabnet_model.predict(X_test)
        y_pred_prob = tabnet_model.predict_proba(X_test)[:,1]

        y_pred_train = tabnet_model.predict(X_train)
        y_proba_train = tabnet_model.predict_proba(X_train)[:, 1]

        y_trues.append(y_test)
        y_preds_proba.append(y_pred_prob)
        
        # Compute metrics
        accuracy_train = accuracy_score(y_train, y_pred_train)
        accuracy_val = accuracy_score(y_test, y_pred_val)
        auroc = roc_auc_score(y_test, y_pred_prob)
        cm = confusion_matrix(y_test, y_pred_val)
    
        # Store results
        accuracies.append(accuracy_val)
        accuracies_train.append(accuracy_train)
        
        aurocs.append(auroc)
        conf_mats.append(cm)

        del tabnet_model
        with torch.no_grad():
            torch.cuda.empty_cache()
        gc.collect()
    
    # Compute mean metrics across folds
    mean_accuracy = np.mean(accuracies)
    mean_accuracy_train = np.mean(accuracies_train)
    
    mean_auroc = np.mean(aurocs)
    total_cm = np.sum(conf_mats, axis=0)  # Summing up all confusion matrices
    
    
    results = {
        'accuracy_val':mean_accuracy,
        'accuracy_train':mean_accuracy_train,
        'auroc':mean_auroc,
        'cm':total_cm,
        'y_proba':np.hstack(y_preds_proba),
        'y_true':np.hstack(y_trues),
    }
    
    return results

In [ ]:
def weighted_fusion(prob_model1, prob_model2, weight_model1, weight_model2):
    fused_probs = (weight_model1 * prob_model1) + (weight_model2 * prob_model2)
    return fused_probs

In [ ]:
def compute_weight(acc1, acc2, alpha):
    return np.exp(alpha * acc1) / (np.exp(alpha * acc1) + np.exp(alpha * acc2))

In [ ]:
def compute_fusion_results(results_cnv, results_rna):
    
    luad_prob_cnv = results_cnv['y_proba']  # CNV model probabilities for LUAD. No need to use LUSC probs, as you will compute fused_probs based on class 1 only
    luad_prob_rna = results_rna['y_proba']  # RNAseq model probabilities for LUAD
    
    acc_cnv = results_cnv['accuracy_train']
    acc_rna = results_rna['accuracy_train']

    k = 10
    alpha = np.abs(np.exp(acc_cnv) - np.exp(acc_rna))*k  # Inline computation of alpha; k = 10 is the scaling factor
    
    weight_cnv = compute_weight(acc_cnv, acc_rna, alpha)
    weight_rna = compute_weight(acc_rna, acc_cnv, alpha)
    
    fused_probs = weighted_fusion(luad_prob_cnv, luad_prob_rna, weight_cnv, weight_rna)
    fused_preds = (fused_probs >= 0.5).astype(int)

    return fused_probs, fused_preds

In [ ]:
def dump_all_results(results, fname, seed, model):
    
    # Extract confusion matrix values
    TN, FP, FN, TP = results['cm'].ravel()
    
    # Create results DataFrame
    results_df = pd.DataFrame({
        "Accuracy": [results['accuracy_val']],
        "Mean AUROC": [results['auroc']],
        "TN": [TN],
        "FP": [FP],
        "FN": [FN],
        "TP": [TP]
    })
    
    # results_df.to_csv(f"Z:/multiomics based manuscript/results_for_xena_rna_cnv_only/{fname}_{seed}.csv", index=False) ## Print results
    results_df.to_csv(f"Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/training_cohort/RESULTS/{model}/{fname}_{seed}.csv", index=False) ## Print results

In [ ]:
## classificaiton using XGBoost

for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df_rna_xena = df_rna.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_cnv_xena = df_cnv.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get XGBoost results on RNASeq
    results_rna = perform_classification(_df_rna_xena, seed)

    ## get XGBoost results on CNV
    results_cnv = perform_classification(_df_cnv_xena, seed)

    print(f"RNA results for seed: {seed}:\t Acc : {results_rna['accuracy_val']} \t AUROC: {results_rna['auroc']}")
    print(f"CNV results for seed: {seed}:\t Acc : {results_cnv['accuracy_val']} \t AUROC: {results_cnv['auroc']}")
    
    fused_probs, fused_preds = compute_fusion_results(results_cnv, results_rna)

    ## now you can use y_true from any dataset -- cnv or rna, as the order of y_true is same in both
    fused_accuracy = accuracy_score(results_cnv['y_true'], fused_preds)
    fused_auroc = roc_auc_score(results_cnv['y_true'], fused_probs)
    fused_cm = confusion_matrix(results_cnv['y_true'], fused_preds)
    
    results_fusion = {
        'accuracy_val':fused_accuracy,
        'auroc':fused_auroc,
        'cm':fused_cm,
    }

    print(f"FUSION results for seed: {seed}:\t Acc : {results_fusion['accuracy_val']} \t AUROC : {results_fusion['auroc']}")

    dump_all_results(results_cnv, 'results_cnv', seed, 'XGB')
    dump_all_results(results_rna, 'results_rna', seed, 'XGB')
    dump_all_results(results_fusion, 'results_fusion', seed, 'XGB')

    print("--------------------------------------------------------------------------------")
    

In [ ]:
## classification using MLP

for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df_rna_xena = df_rna.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_cnv_xena = df_cnv.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get MLP results on RNASeq
    results_rna = perform_classification_mlp(_df_rna_xena, seed)

    ## get MLP results on CNV
    results_cnv = perform_classification_mlp(_df_cnv_xena, seed)

    print(f"RNA results for seed: {seed}:\t Acc : {results_rna['accuracy_val']} \t AUROC: {results_rna['auroc']}")
    print(f"CNV results for seed: {seed}:\t Acc : {results_cnv['accuracy_val']} \t AUROC: {results_cnv['auroc']}")
    
    fused_probs, fused_preds = compute_fusion_results(results_cnv, results_rna)

    ## now you can use y_true from any dataset -- cnv or rna, as the order of y_true is same in both
    fused_accuracy = accuracy_score(results_cnv['y_true'], fused_preds)
    fused_auroc = roc_auc_score(results_cnv['y_true'], fused_probs)
    fused_cm = confusion_matrix(results_cnv['y_true'], fused_preds)
    
    results_fusion = {
        'accuracy_val':fused_accuracy,
        'auroc':fused_auroc,
        'cm':fused_cm,
    }

    print(f"FUSION results for seed: {seed}:\t Acc : {results_fusion['accuracy_val']} \t AUROC : {results_fusion['auroc']}")

    dump_all_results(results_cnv, 'results_cnv', seed, 'MLP')
    dump_all_results(results_rna, 'results_rna', seed, 'MLP')
    dump_all_results(results_fusion, 'results_fusion', seed, 'MLP')

    print("--------------------------------------------------------------------------------")
    

In [ ]:
# classification using SVC

for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df_rna_xena = df_rna.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_cnv_xena = df_cnv.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get SVC results on RNASeq
    results_rna = perform_classification_svc(_df_rna_xena, seed)

    ## get SVC results on CNV
    results_cnv = perform_classification_svc(_df_cnv_xena, seed)

    print(f"RNA results for seed: {seed}:\t Acc : {results_rna['accuracy_val']} \t AUROC: {results_rna['auroc']}")
    print(f"CNV results for seed: {seed}:\t Acc : {results_cnv['accuracy_val']} \t AUROC: {results_cnv['auroc']}")
    
    fused_probs, fused_preds = compute_fusion_results(results_cnv, results_rna)

    ## now you can use y_true from any dataset -- cnv or rna, as the order of y_true is same in both
    fused_accuracy = accuracy_score(results_cnv['y_true'], fused_preds)
    fused_auroc = roc_auc_score(results_cnv['y_true'], fused_probs)
    fused_cm = confusion_matrix(results_cnv['y_true'], fused_preds)
    
    results_fusion = {
        'accuracy_val':fused_accuracy,
        'auroc':fused_auroc,
        'cm':fused_cm,
    }

    print(f"FUSION results for seed: {seed}:\t Acc : {results_fusion['accuracy_val']} \t AUROC : {results_fusion['auroc']}")

    dump_all_results(results_cnv, 'results_cnv', seed, 'SVC')
    dump_all_results(results_rna, 'results_rna', seed, 'SVC')
    dump_all_results(results_fusion, 'results_fusion', seed, 'SVC')

    print("--------------------------------------------------------------------------------")
    

In [ ]:
# classification using TABNET

import torch
from pytorch_tabnet.tab_model import TabNetClassifier

for seed in range(1,6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct

    _df_rna_xena = df_rna.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_cnv_xena = df_cnv.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    ## get TABNET results on RNASeq
    results_rna = perform_tabnet(_df_rna_xena, seed)

    ## get TABNET results on CNV
    results_cnv = perform_tabnet(_df_cnv_xena, seed)

    print(f"RNA results for seed: {seed}:\t Acc : {results_rna['accuracy_val']} \t AUROC: {results_rna['auroc']}")
    print(f"CNV results for seed: {seed}:\t Acc : {results_cnv['accuracy_val']} \t AUROC: {results_cnv['auroc']}")
    
    fused_probs, fused_preds = compute_fusion_results(results_cnv, results_rna)

    ## now you can use y_true from any dataset -- cnv or rna, as the order of y_true is same in both
    fused_accuracy = accuracy_score(results_cnv['y_true'], fused_preds)
    fused_auroc = roc_auc_score(results_cnv['y_true'], fused_probs)
    fused_cm = confusion_matrix(results_cnv['y_true'], fused_preds)
    
    results_fusion = {
        'accuracy_val':fused_accuracy,
        'auroc':fused_auroc,
        'cm':fused_cm,
    }

    print(f"FUSION results for seed: {seed}:\t Acc : {results_fusion['accuracy_val']} \t AUROC : {results_fusion['auroc']}")

    dump_all_results(results_cnv, 'results_cnv', seed, 'TABNET')
    dump_all_results(results_rna, 'results_rna', seed, 'TABNET')
    dump_all_results(results_fusion, 'results_fusion', seed, 'TABNET')

    print("--------------------------------------------------------------------------------")
    